# N7 - N5 × RealMLP 확률 블렌드 (0.4 / 0.6)

- 우리 N5(HGB+TE 10fold×7seed, OOF 0.95045) + 공개 RealMLP(OOF 0.95064) 소프트 블렌드
- **가중치 근거**: OOF 행 5분할 중첩검증 — honest blend 0.95080 (vs N5 +0.00034)
- 가중치 40/60이 비퇴화(non-degenerate)로 잡힌 유일한 조합 (다른 모든 블렌드는 몰빵 수렴으로 기각했음)
- 이 노트북 안에서 OOF 검증을 재계산해 근거를 재현함

In [ ]:
import numpy as np, pandas as pd, glob, os
from sklearn.metrics import balanced_accuracy_score
from sklearn.model_selection import KFold

comp = sorted(glob.glob('/kaggle/input/**/train.csv', recursive=True))
assert comp, 'competition data not mounted'
DATA = os.path.dirname(comp[0])
CLASSES = ['at-risk','unhealthy','fit']
train = pd.read_csv(f'{DATA}/train.csv', usecols=['health_condition'])
y = train['health_condition'].map({c:i for i,c in enumerate(CLASSES)}).to_numpy()
ss = pd.read_csv(f'{DATA}/sample_submission.csv')

n5_oof_p  = sorted(glob.glob('/kaggle/input/**/n5_oof.npy',  recursive=True))
n5_test_p = sorted(glob.glob('/kaggle/input/**/n5_test.npy', recursive=True))
rm_oof_p  = sorted(glob.glob('/kaggle/input/**/oof_preds.csv',  recursive=True))
rm_test_p = sorted(glob.glob('/kaggle/input/**/test_preds.csv', recursive=True))
assert n5_oof_p and rm_oof_p and n5_test_p and rm_test_p, 'sources missing'
hgb, hgb_t = np.load(n5_oof_p[0]), np.load(n5_test_p[0])
ro = pd.read_csv(rm_oof_p[0]).sort_values('id')
rt = pd.read_csv(rm_test_p[0]).set_index('id').loc[ss['id']].reset_index()
rmlp   = ro[['at-risk','unhealthy','fit']].to_numpy()   # 컬럼 순서 정렬
rmlp_t = rt[['at-risk','unhealthy','fit']].to_numpy()
print('N5 solo  :', round(balanced_accuracy_score(y, hgb.argmax(1)), 5))
print('RMLP solo:', round(balanced_accuracy_score(y, rmlp.argmax(1)), 5))

In [ ]:
# 중첩검증으로 가중치 재산출 (재현 가능성)
P = np.stack([hgb, rmlp]); grid = np.linspace(0, 1, 41)
kf = KFold(5, shuffle=True, random_state=0)
honest, base, ws = [], [], []
for tr, va in kf.split(y):
    best = (-1, None)
    for w in grid:
        s = balanced_accuracy_score(y[tr], (w*P[0][tr] + (1-w)*P[1][tr]).argmax(1))
        if s > best[0]: best = (s, w)
    ws.append(best[1])
    honest.append(balanced_accuracy_score(y[va], (best[1]*P[0][va] + (1-best[1])*P[1][va]).argmax(1)))
    base.append(balanced_accuracy_score(y[va], P[0][va].argmax(1)))
W = float(np.mean(ws))
print(f'w(N5)={W:.3f} | honest={np.mean(honest):.5f} | vs N5 {np.mean(honest)-np.mean(base):+.5f}')

In [ ]:
final = W * hgb_t + (1 - W) * rmlp_t
inv = {i: c for i, c in enumerate(CLASSES)}
sub = pd.DataFrame({'id': ss['id'], 'health_condition': [inv[i] for i in final.argmax(1)]})
sub.to_csv('submission.csv', index=False)
assert len(sub) == len(ss) and sub['health_condition'].isna().sum() == 0
print('submission.csv | sanity OK |', sub['health_condition'].value_counts().to_dict())